In [29]:
from SteerEnergyStorage.Formulations.ElectrodeFormulations import ElectrodeFormulation
from SteerEnergyStorage.Constructions.Electrodes import Cathode, Anode
from SteerEnergyStorage.Formulations.Stacks import Stack
from SteerEnergyStorage.Constructions.Cells import StackedPrismaticCell
from SteerEnergyStorage.Materials.ElectrodeMaterials import CathodeMaterial, AnodeMaterial, Binder, ConductiveAdditive
from SteerEnergyStorage.Materials.CurrentCollectors import CurrentCollector
from SteerEnergyStorage.Materials.Separators import Separator
from SteerEnergyStorage.Materials.Electrolytes import Electrolyte
from SteerEnergyStorage.Constructions.Containers import PrismaticCase, PrismaticLid, PrismaticShell

import pandas as pd
import plotly.express as px
import numpy as np

In [ ]:
n = 10000

In [ ]:
# construct cathodes

cathodes = []

for i in range(0, n):

    # create formulation materials
    cathode_active_material = CathodeMaterial(name=f"Faradion_Gen2_4.25V", specific_cost=11.26, density=4)
    cathode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9, name=f"SuperC65_cathode_{i}")
    cathode_binder = Binder(specific_cost=15, density = 1.7)

    # create formulation with noisy fractions
    active_material_fraction = np.random.normal(89, 1)
    binder_fraction = np.random.normal(5, 1)
    conductive_additive_fraction = 100 - active_material_fraction - binder_fraction

    cathode_formulation = ElectrodeFormulation(active_materials={cathode_active_material: active_material_fraction},
                                               binder={cathode_binder: binder_fraction},
                                               conductive_additive={cathode_conductive_additive: conductive_additive_fraction}
                                               )

    # create current collector
    cathode_current_collector = CurrentCollector(formula="Al", thickness=15, length=15, width=11.8, bare_tab_area=16)

    # create cathode
    cathode = Cathode(formulation=cathode_formulation,
                      mass_loading=np.random.normal(10.38, 0.1),
                      current_collector=cathode_current_collector,
                      calender_density=np.random.normal(2.6, 0.1)
                      )
    
    # append cathode to list
    cathodes.append(cathode) 


In [32]:
# construct anode

anodes = []

for i in range(0, n):
    # create formulation materials
    anode_active_material = AnodeMaterial(name="faradion_hc", specific_cost=14.27, density=1.5)
    anode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9)
    anode_binder = Binder(specific_cost=10, density=1.7)

    # create formulation with noisy fractions
    active_material_fraction = np.random.normal(88, 1)
    binder_fraction = np.random.normal(3, 1)
    conductive_additive_fraction = 100 - active_material_fraction - binder_fraction

    # create formulation
    anode_formulation = ElectrodeFormulation(active_materials={anode_active_material: active_material_fraction},
                                            binder={anode_binder: binder_fraction},
                                            conductive_additive={anode_conductive_additive: conductive_additive_fraction})

    # create current collector
    anode_current_collector = CurrentCollector(formula="Cu", thickness=4, length=15, width=11.8, bare_tab_area=7.55)

    # create anode
    anode = Anode(formulation=anode_formulation,
                  mass_loading=np.random.normal(5.20, 0.1),
                  current_collector=anode_current_collector,
                  calender_density=0.85)
    
    anodes.append(anode)


In [33]:
# Other cell components

seperators = []
stacks = []
for i in range(0, n):
    # construct seperator
    seperator = Separator(thickness=16, areal_cost=0.9, density=0.4, slit_width=12, porosity=47, fold_length=18.6)
    seperators.append(seperator)


In [34]:

# make electrolyte
electrolyte = Electrolyte(name="LiPF6", specific_cost=8.94, density=1.2)

cases = []
stacks = []
for i in range(0, n):
    prismatic_lid = PrismaticLid(cost=0.05, mass=10, external_width=1.3, internal_width=0.8)
    prismatic_shell = PrismaticShell(cost=0.15, mass=60, internal_length=19, internal_width=13, internal_height=0.5, wall_thickness=0.5)
    prismatic_case = PrismaticCase(lid=prismatic_lid, shell=prismatic_shell)
    cases.append(prismatic_case)

    stack = prismatic_case.get_optimized_stack(cathode=cathodes[i], anode=anodes[i], separator=seperators[i])
    stacks.append(stack)


In [35]:
cells = []
for i in range(0, n):
    cell = StackedPrismaticCell(stack=stacks[i],
                                prismatic_case=cases[i],
                                electrolyte=electrolyte,
                                electrolyte_overfill=10,
                                reversible_capacity=10,
                                irreversible_capacity=1)
    
    cells.append(cell)

In [36]:
cells[0].get_capacity_voltage_plot(width=900, height=500)

In [37]:
figure = cells[0].get_mass_breakdown_plot(width=900, height=400)
figure.show()

In [38]:
figure = cells[0].get_cost_breakdown_plot(width=900, height=400)
figure.show()

In [39]:
# look at the distribution of the cost of the cells

plot_data = pd.DataFrame({
    'cell_name': [c.name for c in cells],
    'cell_cost': [c.normalized_cost for c in cells]
})

px.histogram(plot_data, x='cell_cost', title='Distribution of cell costs', nbins=50, template='presentation', 
             labels={'cell_cost': 'Cost $/kWh'}, width=700, height=400).show()

In [40]:
# look at the distribution of the cost of the cell stacks

plot_data = pd.DataFrame({
    'cell_name': [c.name for c in cells],
    'stack_cost': [c.cost_breakdown[s] for c, s in zip(cells, stacks)]
})

px.histogram(plot_data, x='stack_cost', title='Distribution of cell costs', nbins=50, template='presentation', 
             labels={'stack_cost': 'Stack cost $'}, width=700, height=400).show()

In [41]:
# look at the distribution of the masses of the cells

plot_data = pd.DataFrame({
    'cell_name': [c.name for c in cells],
    'cell_mass': [c.mass for c in cells]
})

px.histogram(plot_data, x='cell_mass', title='Distribution of cell masses', nbins=50, template='presentation', 
             labels={'cell_mass': 'Mass [g]'}, width=700, height=400).show()

In [42]:
plot_data = pd.DataFrame({
    'cell_name': [c.name for c in cells],
    'number_of_stack_layers': [c.stack.n_stacks for c in cells]
})


px.histogram(plot_data, x='number_of_stack_layers', title='Distribution of cell masses', nbins=50, template='presentation', 
             labels={'cell_mass': 'Mass [g]'}, width=700, height=400).show()